# 🚗 Assistente Automotivo por Voz

Projeto de assistente inteligente com reconhecimento de voz voltado para entusiastas e proprietários de veículos.  
Fale sua dúvida automotiva e receba uma resposta de um especialista — tudo por voz.

**Tecnologias utilizadas:**
- [Whisper (OpenAI)](https://github.com/openai/whisper) — transcrição de voz para texto (roda localmente, sem custo)
- [Groq](https://console.groq.com) — geração de respostas inteligentes com LLaMA (gratuito)
- [gTTS](https://pypi.org/project/gTTS/) — síntese de voz (text-to-speech)
- Google Colab — ambiente de execução

**Exemplos de perguntas:**
- *"Qual a diferença entre motor aspirado e turbinado?"*
- *"Meu carro está fazendo barulho ao frear, o que pode ser?"*
- *"Me fala sobre o motor rotativo do RX-7"*
- *"Com quantos km eu devo trocar o óleo?"*

## Etapa 1 — Instalação das dependências

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install groq -q
!pip install gTTS -q

## Etapa 2 — Configuração

Para gerar uma chave gratuita do Groq acesse: https://console.groq.com → API Keys → Create API Key

In [ ]:
import os

os.environ['GROQ_API_KEY'] = 'SUA_CHAVE_AQUI'

# Idioma utilizado na transcrição e na resposta em voz
language = 'pt'

## Etapa 3 — Gravação do áudio

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('🎙️ Ouvindo... Fale sua dúvida automotiva!\n')
record_file = record()

display(Audio(record_file, autoplay=False))

## Etapa 4 — Transcrição com Whisper

In [ ]:
import whisper

model = whisper.load_model("small")

result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]

print(f'🗣️ Você disse: {transcription}')

## Etapa 5 — Resposta do Especialista Automotivo com Groq

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

system_prompt = """
Você é um especialista automotivo experiente com profundo conhecimento em:
- Mecânica geral e manutenção preventiva
- Motores (aspirados, turbo, híbridos, elétricos)
- Diagnóstico de problemas e falhas
- Performance e modificações
- Cultura automotiva e história de modelos icônicos

Responda de forma clara, direta e acessível, como um mecânico de confiança conversando com um amigo.
Evite respostas muito longas — seja objetivo e prático.
Responda sempre em português brasileiro.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": transcription}
    ]
)

groq_response = response.choices[0].message.content
print(f'🔧 Especialista: {groq_response}')

## Etapa 6 — Resposta em Voz com gTTS

In [ ]:
from gtts import gTTS

gtts_object = gTTS(text=groq_response, lang=language, slow=False)

response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

print('🔊 Reproduzindo resposta...\n')
display(Audio(response_audio, autoplay=True))